# ANOVA + Tukey HSD — LMS Cache Strategy Analysis

Methodology: One-Way ANOVA with alpha = 0.05 and Tukey HSD post-hoc analysis for valid 1500 VU benchmark iterations only. The 1500 VU level is the main inferential comparison point because it keeps the workload high while excluding saturated rows where unexpected error rate exceeds 5%.

The 2000 VU level remains descriptive evidence for scalability and saturation behavior, especially when write-heavy requests exceed the validity threshold.


In [ ]:
%pip install scipy statsmodels seaborn --quiet

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)


## Data Load

Upload nova-input.csv from enchmark-results-combined/. Optionally upload metrics-summary.csv too; it is only used for descriptive 2000 VU context.


In [ ]:
try:
    from google.colab import files

    uploaded = files.upload()
except ImportError:
    uploaded = {}

anova_file = 'anova-input.csv'
metrics_summary_file = 'metrics-summary.csv'

if anova_file not in uploaded:
    print('Using local anova-input.csv if available in the current runtime.')

anova_df = pd.read_csv(anova_file)
metrics_summary_df = pd.read_csv(metrics_summary_file) if metrics_summary_file in uploaded else None

print(f'anova-input rows: {len(anova_df):,}')
display(anova_df.head())


## Data Validation

The validity table counts valid and saturated iterations by Redis mode, workload, VU level, and cache strategy. Valid rows use the thesis threshold unexpected_error_rate_pct <= 5.


In [ ]:
required_columns = {
    'redis_mode',
    'strategy',
    'scenario',
    'concurrent_users',
    'iteration',
    'avg_ms',
    'p95_ms',
    'p99_ms',
    'throughput_rps',
    'cache_hit_ratio_pct',
    'error_rate_pct',
    'unexpected_error_rate_pct',
    'controlled_error_rate_pct',
    'validity_status',
}
missing_columns = required_columns.difference(anova_df.columns)
if missing_columns:
    raise ValueError(f'Missing required columns: {sorted(missing_columns)}')

strategies = ['no-cache', 'cache-aside', 'read-through', 'write-through']
analysis_metrics = [
    'avg_ms',
    'p95_ms',
    'p99_ms',
    'throughput_rps',
    'cache_hit_ratio_pct',
    'unexpected_error_rate_pct',
]
target_vu = 1500
saturation_vu = 2000
alpha = 0.05

anova_df['validity_status'] = np.where(anova_df['unexpected_error_rate_pct'] <= 5, 'valid', 'saturated')

validity_summary = (
    anova_df
    .groupby(['redis_mode', 'scenario', 'concurrent_users', 'strategy', 'validity_status'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
for column in ['valid', 'saturated']:
    if column not in validity_summary.columns:
        validity_summary[column] = 0
validity_summary['total_iterations'] = validity_summary['valid'] + validity_summary['saturated']
validity_summary = validity_summary[
    ['redis_mode', 'scenario', 'concurrent_users', 'strategy', 'valid', 'saturated', 'total_iterations']
]
validity_summary.to_csv('validity-summary.csv', index=False)

df_1500 = anova_df[
    (anova_df['concurrent_users'] == target_vu)
    & (anova_df['validity_status'] == 'valid')
].copy()

group_counts_1500 = (
    df_1500
    .groupby(['redis_mode', 'scenario', 'strategy'])
    .size()
    .reset_index(name='valid_iterations')
    .sort_values(['redis_mode', 'scenario', 'strategy'])
)

print(f'1500 VU valid rows: {len(df_1500):,}')
print('validity-summary.csv exported')
display(validity_summary)
display(group_counts_1500)


## One-Way ANOVA at 1500 VU

Each ANOVA compares cache strategies inside one fixed Redis mode, workload, and metric. A condition is skipped when one or more strategies has fewer than 3 valid iterations.


In [ ]:
def condition_subset(dataframe, redis_mode, scenario):
    return dataframe[
        (dataframe['redis_mode'] == redis_mode)
        & (dataframe['scenario'] == scenario)
        & (dataframe['strategy'].isin(strategies))
    ].copy()


def strategy_groups(dataframe, metric):
    groups = []
    counts = {}
    for strategy in strategies:
        values = dataframe.loc[dataframe['strategy'] == strategy, metric].dropna().astype(float).to_numpy()
        groups.append(values)
        counts[strategy] = len(values)
    return groups, counts


def run_anova_for_condition(dataframe, redis_mode, scenario, metric):
    subset = condition_subset(dataframe, redis_mode, scenario)
    groups, counts = strategy_groups(subset, metric)

    result = {
        'redis_mode': redis_mode,
        'scenario': scenario,
        'metric': metric,
        'target_vu': target_vu,
        'alpha': alpha,
        'n_no_cache': counts['no-cache'],
        'n_cache_aside': counts['cache-aside'],
        'n_read_through': counts['read-through'],
        'n_write_through': counts['write-through'],
        'f_statistic': np.nan,
        'p_value': np.nan,
        'is_significant': False,
        'decision': 'skipped',
        'skip_reason': '',
    }

    if any(count < 3 for count in counts.values()):
        result['skip_reason'] = 'each strategy needs at least 3 valid iterations'
        return result

    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        f_statistic, p_value = stats.f_oneway(*groups)

    result['f_statistic'] = f_statistic
    result['p_value'] = p_value

    if np.isfinite(p_value) and p_value < alpha:
        result['is_significant'] = True
        result['decision'] = 'significant'
    elif np.isfinite(p_value):
        result['decision'] = 'not significant'
    else:
        result['decision'] = 'not testable'
        result['skip_reason'] = 'ANOVA returned NaN, usually because all compared values are constant'

    return result


anova_rows = []
for redis_mode in sorted(df_1500['redis_mode'].dropna().unique()):
    for scenario in sorted(df_1500['scenario'].dropna().unique()):
        for metric in analysis_metrics:
            anova_rows.append(run_anova_for_condition(df_1500, redis_mode, scenario, metric))

anova_results = pd.DataFrame(anova_rows)
anova_results.to_csv('anova-results-1500vu.csv', index=False)

print('anova-results-1500vu.csv exported')
display(anova_results)


## Tukey HSD Post-Hoc

Tukey HSD is only run for ANOVA rows that are significant and have at least 3 valid iterations for every strategy.


In [ ]:
def run_tukey_for_condition(dataframe, redis_mode, scenario, metric):
    subset = condition_subset(dataframe, redis_mode, scenario)
    subset = subset[['strategy', metric]].dropna().copy()
    subset[metric] = subset[metric].astype(float)

    tukey = pairwise_tukeyhsd(endog=subset[metric], groups=subset['strategy'], alpha=alpha)
    tukey_table = pd.DataFrame(tukey._results_table.data[1:], columns=tukey._results_table.data[0])
    tukey_table.insert(0, 'redis_mode', redis_mode)
    tukey_table.insert(1, 'scenario', scenario)
    tukey_table.insert(2, 'metric', metric)
    tukey_table.insert(3, 'target_vu', target_vu)
    return tukey_table


tukey_tables = []
for row in anova_results.itertuples(index=False):
    if row.is_significant:
        tukey_tables.append(run_tukey_for_condition(df_1500, row.redis_mode, row.scenario, row.metric))

if tukey_tables:
    tukey_results = pd.concat(tukey_tables, ignore_index=True)
else:
    tukey_results = pd.DataFrame(
        columns=['redis_mode', 'scenario', 'metric', 'target_vu', 'group1', 'group2', 'meandiff', 'p-adj', 'lower', 'upper', 'reject']
    )

tukey_results.to_csv('tukey-results-1500vu.csv', index=False)

print('tukey-results-1500vu.csv exported')
display(tukey_results)


## 2000 VU Saturation Summary

The 2000 VU level is summarized descriptively only. Use it to discuss scalability limits and saturation, not as the primary ANOVA/Tukey comparison.


In [ ]:
df_2000 = anova_df[anova_df['concurrent_users'] == saturation_vu].copy()

saturation_summary = (
    df_2000
    .groupby(['redis_mode', 'scenario', 'strategy'])
    .agg(
        iterations=('iteration', 'nunique'),
        valid_iterations=('validity_status', lambda values: (values == 'valid').sum()),
        saturated_iterations=('validity_status', lambda values: (values == 'saturated').sum()),
        mean_error_rate_pct=('error_rate_pct', 'mean'),
        max_error_rate_pct=('error_rate_pct', 'max'),
        mean_unexpected_error_rate_pct=('unexpected_error_rate_pct', 'mean'),
        max_unexpected_error_rate_pct=('unexpected_error_rate_pct', 'max'),
        mean_avg_ms=('avg_ms', 'mean'),
        mean_p95_ms=('p95_ms', 'mean'),
        mean_p99_ms=('p99_ms', 'mean'),
        mean_throughput_rps=('throughput_rps', 'mean'),
        mean_cache_hit_ratio_pct=('cache_hit_ratio_pct', 'mean'),
    )
    .reset_index()
    .sort_values(['redis_mode', 'scenario', 'strategy'])
)

display(saturation_summary)

if metrics_summary_df is not None:
    print('Uploaded metrics-summary.csv, 2000 VU descriptive rows:')
    display(metrics_summary_df[metrics_summary_df['concurrent_users'] == saturation_vu])


## 1500 VU Boxplots

These plots show valid 1500 VU latency distributions by strategy, separated by Redis mode and workload.


In [ ]:
latency_metrics = ['avg_ms', 'p95_ms', 'p99_ms']
plot_df = df_1500.melt(
    id_vars=['redis_mode', 'scenario', 'strategy', 'iteration'],
    value_vars=latency_metrics,
    var_name='metric',
    value_name='latency_ms',
)

if plot_df.empty:
    print('No valid 1500 VU rows available for boxplots.')
else:
    for metric in latency_metrics:
        metric_df = plot_df[plot_df['metric'] == metric]
        grid = sns.catplot(
            data=metric_df,
            x='strategy',
            y='latency_ms',
            row='scenario',
            col='redis_mode',
            kind='box',
            order=strategies,
            sharey=False,
            height=3.2,
            aspect=1.35,
        )
        grid.set_axis_labels('Strategy', f'{metric} (ms)')
        grid.set_titles(row_template='{row_name}', col_template='{col_name}')
        for axis in grid.axes.flat:
            axis.tick_params(axis='x', rotation=25)
        plt.tight_layout()
        plt.show()


## Interpretation Helper

Use these outputs to keep the thesis discussion consistent: 1500 VU is the main statistical comparison, while 2000 VU supports the saturation/scalability discussion.


In [ ]:
significant_rows = anova_results[anova_results['is_significant']]
skipped_rows = anova_results[anova_results['decision'] == 'skipped']
saturated_2000 = saturation_summary[saturation_summary['saturated_iterations'] > 0]

print(f'1500 VU valid rows analyzed: {len(df_1500):,}')
print(f'Significant ANOVA results: {len(significant_rows):,} of {len(anova_results):,}')
print(f'Skipped ANOVA results: {len(skipped_rows):,}')
print(f'2000 VU strategy conditions with saturation: {len(saturated_2000):,}')

if not significant_rows.empty:
    print()
    print('Significant 1500 VU ANOVA rows:')
    display(significant_rows[['redis_mode', 'scenario', 'metric', 'f_statistic', 'p_value']])

if not saturated_2000.empty:
    print()
    print('2000 VU saturation conditions:')
    display(saturated_2000[['redis_mode', 'scenario', 'strategy', 'saturated_iterations', 'mean_unexpected_error_rate_pct', 'mean_p99_ms']])


## Exported Files

Running this notebook exports:

- alidity-summary.csv
- nova-results-1500vu.csv
- 	ukey-results-1500vu.csv
